In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import cv2

import torch
from torch import nn

import os
import gc
from tqdm import tqdm

### Setting : Test Loss

#### A. Load Test Loss during the Training

In [ ]:
losses = np.load(r"C:\Users\dlgkr\Downloads\train0319_1_losses.npz")
train_loss = losses['train_loss']
test_loss = losses['test_loss']
indv_losses = losses['indv_losses'][:, 1:]
train_loss : np.ndarray
test_loss : np.ndarray
indv_losses : np.ndarray

#### B. Calculate Test Loss from Saved Model

In [ ]:
class QValueNet_CNN_B1(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, activation=nn.ReLU, dropout=0.3):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.activation = activation

        # R_arr encoders (input: [B, C, 40, 20])
        self.r_arr_encoder1 = nn.Sequential(
            nn.Conv2d(1, 8, (9, 5)),  # -> [B, 8, 40, 20] # 1 channel / assumed input is already done padding=1 #(1, 16, 3)
            self.activation(),
            #nn.MaxPool2d(2)  # -> [B, 8, 20, 10] #deleted (260108) : to remain size 40x20
        )

        self.r_arr_encoder2 = nn.Sequential(
            nn.Conv2d(8, 16, (5, 3)),  # assumed input is already done padding=1 #(16, 32, 3)
            self.activation(), # -> [B, 16, 20, 10] # -> [B, 16, 40, 20]

            # for preserving spatial structure, using size 1 kernal instead MLP
            nn.Conv2d(16, 64, 1), # -> [B, 64, 20, 10] # -> [B, 16, 40, 20]
            self.activation(),
            #nn.AdaptiveAvgPool2d(1) # -> [B, 64, 1, 1]#deleted (260108) : to remain size 40x20
        )

        # Info encoder (input: [B, 1, 6])
        self.info_encoder = nn.Sequential(
            nn.Linear(6, 32),
            self.activation(),
            nn.Linear(32, 64) # -> [B, 1, 64]
        )

        # RL encoder (input: [B, 1, 4])
        self.rl_encoder = nn.Sequential(
            nn.Linear(4, 32),
            self.activation(),
            nn.Linear(32, 64) # -> [B, 1, 64]
        )

        # Lightcurves encoder (input: [B, 2, 100])
        self.lc_encoder1 = nn.Sequential(
            nn.Conv1d(2, 16, kernel_size=15),
            self.activation(),
            nn.MaxPool1d(2),   # -> [B, 16, 50]
        )

        self.lc_encoder2 = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=9),
            self.activation(), # -> [B, 32, 50]

            nn.Conv1d(32, 64, 1), # -> [B, 64, 50]
            self.activation(),
            nn.AdaptiveAvgPool1d(8), # -> [B, 64, 8]

            nn.Flatten(1, -1), # -> [B, 512]
            nn.Linear(64*8, 64), # -> [B, 64]
            self.activation(),
            nn.Dropout(dropout)
        )

        # Fusion & Head
        self.head = nn.Sequential(
            nn.Linear(64 + 64 + 64 + 64, self.hidden_dim),
            self.activation(),
            nn.Dropout(dropout),

            nn.Linear(self.hidden_dim, self.hidden_dim),
            self.activation(),
            nn.Dropout(dropout),

            nn.Linear(self.hidden_dim, 256),
            self.activation(),
            nn.Dropout(dropout),

            nn.Linear(256, 1)  # e.g., class count or regression value
        )

    def r_padding(self, x, pad=(1, 1)):
        N, C, H, W = x.shape
        pad_H = pad[0]
        pad_W = pad[1]

        out = torch.full((N, C, H + 2*pad_H, W + 2*pad_W), fill_value=0.0, dtype=x.dtype, device=x.device)
        out[:, :, pad_H:pad_H+H, pad_W:pad_W+W] = x
        out[:, :, :, :pad_W] = torch.roll(torch.flip(out[:, :, :, pad_W:pad_W+pad_W], (-2,)), 20, -1)
        out[:, :, :, -pad_W:] = torch.roll(torch.flip(out[:, :, :, -pad_W-pad_W:-pad_W], (-2,)), 20, -1)
        out[:, :, :pad_H, pad_W:pad_W+W] = x[:, :, -pad_H:, :]
        out[:, :, -pad_H:, pad_W:pad_W+W] = x[:, :, :pad_H, :]
        return out

    def lc_padding(self, x, pad=1):
        N, C, W = x.shape

        out = torch.full((N, C, W + 2*pad), fill_value=0.0, dtype=x.dtype, device=x.device)
        out[:, :, pad:pad+W] = x
        out[:, :, :pad] = x[:, :, -pad:]
        out[:, :, -pad:] = x[:, :, :pad]
        return out

    def shifter(self, img, dx=0, dy=0):
        PI = 3.14159265358979
        img_F = torch.fft.fft2(img)
        N, M = img.shape
        dev = img.device

        ky = torch.fft.fftfreq(N, device=dev)[:, None]
        kx = torch.fft.fftfreq(M, device=dev)[None, :]
        phase = torch.exp(-2j*PI*(kx*dx + ky*dy))
        new_img = torch.fft.ifft2(img_F*phase)
        return new_img.real

    def sphere_latlon_tensor(self, lon, lat, Nlon=40, Nlat=20):
        # Added w/ GPT (26.01.11)
        lon = lon.long()
        lat = lat.long()

        # lon은 항상 주기 wrap
        lon = torch.remainder(lon, Nlon)

        half = Nlon // 2
        m1 = lat < 0
        m2 = (~m1) & (lat >= Nlat)

        lat = torch.where(m1, -lat, lat)
        lon = torch.where(m1, lon + half, lon)

        lat = torch.where(m2, 2*(Nlat-1) - lat, lat)
        lon = torch.where(m2, lon + half, lon)

        # 보정 후에도 다시 wrap
        lon = torch.remainder(lon, Nlon)
        return lon, lat


    def r_a_gather(self, r_arr_feat, lon, lat, size=3):
        # Added w/ GPT (26.01.11)
        if size%2 == 0: raise ValueError("size must be odd")

        B = r_arr_feat.shape[0]
        b_idx = torch.arange(B, device=r_arr_feat.device)

        r_a_elems = []
        for i in range(-size//2, size//2+1):
            for j in range(-size//2, size//2+1):
                lon_temp, lat_temp = self.sphere_latlon_tensor(lon+i, lat+j)
                r_a_elems.append(r_arr_feat[b_idx, :, lon_temp, lat_temp])
        #r_a = torch.cat(r_a_elems, dim=1)
        r_a = r_a_elems[0]
        for i in range(1, len(r_a_elems)): r_a = r_a + r_a_elems[i]
        r_a = r_a / (size**2)

        return r_a

    def forward(self, X):
        if X.dim() == 3 and X.size(1) == 1:
            X = X.squeeze(1)  # [B, input_dim]
        PI = 3.14159265358979

        r_arr = X[..., :800].reshape((X.shape[0], 1, 40, 20))
        lc_target = X[..., 800:900].reshape((X.shape[0], 1, 100))
        lc_pred = X[..., 900:1000].reshape((X.shape[0], 1, 100))
        lc_info = X[..., 1000:1006]
        rl_info = X[..., 1006:]

        #r_arr_feat = torch.transpose(r_arr, -2, -1)
        #r_arr_feat = self.r_padding(r_arr_feat, pad=(4, 2))
        r_arr_feat = self.r_padding(r_arr, pad=(4, 2))
        r_arr_feat = self.r_arr_encoder1(r_arr_feat)
        r_arr_feat = self.r_padding(r_arr_feat, pad=(2, 1))
        r_arr_feat = self.r_arr_encoder2(r_arr_feat)
        #r_arr_feat = torch.squeeze(r_arr_feat, dim=-1)
        #r_arr_feat = torch.squeeze(r_arr_feat, dim=-1)

        ##############################################################
        # select action coord. from feature map (GPT, 260108)
        lon_raw = rl_info[..., 0]  # [B]  (아직 embedding 전, 0~1 가정)
        lat_raw = rl_info[..., 1]  # [B]

        lon_idx = torch.floor(lon_raw * 40).clamp(0, 39).long()
        lat_idx = torch.floor(lat_raw * 20).clamp(0, 19).long()

        # r_arr_feat가 [B, 64, 40, 20]일 때:
        B = r_arr_feat.shape[0]
        b_idx = torch.arange(B, device=r_arr_feat.device)
        r_a = r_arr_feat[b_idx, :, lon_idx, lat_idx]     # [B, 64]
        #r_a = self.r_a_gather(r_arr_feat, lon_idx, lat_idx, size=3) # [B, 64]
        ##############################################################

        lc = torch.cat([lc_target, lc_pred], dim=1)
        lc_feat = self.lc_padding(lc, pad=7)          # [B, 2, 114]
        lc_feat = self.lc_encoder1(lc_feat)           # [B, 16, 50]
        lc_feat = self.lc_padding(lc_feat, pad=4)     # [B, 16, 58]
        lc_feat = self.lc_encoder2(lc_feat)           # [B, 64, 1]
        #lc_feat = torch.squeeze(lc_feat, dim=-1)      # [B, 64]

        info_feat = self.info_encoder(lc_info)
        #info_feat = torch.squeeze(info_feat, dim=1)

        # action direction embedding
        lon_raw = rl_info[..., 0]  # [B]  (아직 embedding 전, 0~1 가정)
        lat_raw = rl_info[..., 1]  # [B]
        action_emb = torch.stack([
            torch.sin(2*PI*lon_raw),
            torch.cos(2*PI*lon_raw),
            torch.sin(PI*lat_raw),
            torch.cos(PI*lat_raw),
        ], dim=1)  # [B,4]
        rl_feat = self.rl_encoder(action_emb)
        #rl_info = torch.unsqueeze(rl_info, dim=1)
        #rl_feat = self.rl_encoder(rl_info)
        #rl_feat = torch.squeeze(rl_feat, dim=1)

        #fusion_feat = torch.cat((r_arr_feat, lc_feat, info_feat, rl_feat), dim=1)
        fusion_feat = torch.cat((r_a, lc_feat, info_feat, rl_feat), dim=1)
        out = self.head(fusion_feat)
        #shift_out = self.shift_head(fusion_feat)

        #self.x_shift = torch.unsqueeze(shift_out[..., 0], dim=1)
        #self.y_shift = torch.unsqueeze(shift_out[..., 1], dim=1)

        #out = self.shifter(out, dx=20*self.x_shift, dy=10*self.y_shift)

        out = 6 * 2 / PI * torch.atan(out/0.8) #out/0.8
        #out = 7 * 2 / PI * torch.atan(1.5 * out)

        return out
    

class RewardMapModifier():
    def __init__(self, extends=(0, 1), blur_coef=(5, 3)):
        self.extends = extends
        self.blur_coef = blur_coef

    def extend_hori(self, reward_map, action_maps):
        left_reward = reward_map[..., :, -int(reward_map.shape[-2]*self.extends[1]/2):, :]
        right_reward = reward_map[..., :, :int(reward_map.shape[-2]*self.extends[1]/2), :]

        if action_maps is not None:
            left_actions = action_maps[..., :, -int(action_maps.shape[-2]*self.extends[1]/2):, :].copy()
            right_actions = action_maps[..., :, :int(action_maps.shape[-2]*self.extends[1]/2), :].copy()
            left_actions[..., :, :, 0] = left_actions[..., :, :, 0] - 1
            right_actions[..., :, :, 0] = right_actions[..., :, :, 0] + 1

        if self.extends[1] != 0:
            extended_reward = np.concatenate((left_reward, reward_map, right_reward), axis=-2)
            extended_actions = np.concatenate((left_actions, action_maps, right_actions), axis=-2) if action_maps is not None else action_maps
        else:
            extended_reward = reward_map
            extended_actions = action_maps

        return extended_reward, extended_actions

    def extend_vert(self, reward_map, action_maps):
        top_reward = np.roll(reward_map[..., :int(reward_map.shape[-3]*self.extends[0]/2), :, :], 20, axis=-2)
        bottom_reward = np.roll(reward_map[..., -int(reward_map.shape[-3]*self.extends[0]/2):, :, :], 20, axis=-2)
        top_reward = np.flip(top_reward, axis=-3)
        bottom_reward = np.flip(bottom_reward, axis=-3)

        if action_maps is not None:
            top_actions = np.flip(action_maps[..., :int(action_maps.shape[-3]*self.extends[0]/2), :, :].copy(), -3)
            bottom_actions = np.flip(action_maps[..., -int(action_maps.shape[-3]*self.extends[0]/2):, :, :].copy(), -3)
            top_actions[..., :, :, 1] = 2*0 - top_actions[..., :, :, 1]
            bottom_actions[..., :, :, 1] = 2*1 - bottom_actions[..., :, :, 1]

        if self.extends[0] != 0:
            extended_reward = np.concatenate((top_reward, reward_map, bottom_reward), axis=-3)
            extended_actions = np.concatenate((top_actions, action_maps, bottom_actions), axis=-3) if action_maps is not None else action_maps
        else:
            extended_reward = reward_map
            extended_actions = action_maps

        return extended_reward, extended_actions

    def blur(self, reward_map):
        #reward_map = 2.5 * np.tan( reward_map * (np.pi/2) / 6 )\n",
        if len(reward_map.shape) == 3:
            reward_map[:, :, 0] = cv2.GaussianBlur(reward_map[:, :, 0], (self.blur_coef[0], self.blur_coef[0]), self.blur_coef[1])
        elif len(reward_map.shape) == 4:
            for i in range(reward_map.shape[0]):
                reward_map[i, :, :, 0] = cv2.GaussianBlur(reward_map[i, :, :, 0], (self.blur_coef[0], self.blur_coef[0]), self.blur_coef[1])
                #max_val = np.max(np.abs(reward_map[i, :, :, 0]))
                #reward_map[i, :, :, 0] = 6 * (2/np.pi) * np.arctan(reward_map[i, :, :, 0]/2) / ((2/np.pi) * np.arctan(max_val/2))
        reward_map = 6 * (2/np.pi) * np.arctan(reward_map/8)
        #reward_map = 6 * 2*(1/(1+np.exp(-reward_map/7)) - 0.5)
        #reward_map = 6 * (2/np.pi) * np.arctan(reward_map/2)
        return reward_map

    def operation(self, reward_map, action_maps, order=['extend_hori', 'extend_vert', 'blur']):
        result_reward = reward_map
        result_action = action_maps
        for op in order:
            if op == 'extend_hori':
                result_reward, result_action = self.extend_hori(result_reward, result_action)
            elif op == 'extend_vert':
                result_reward, result_action = self.extend_vert(result_reward, result_action)
            elif op == 'blur':
                result_reward = self.blur(result_reward)
            else:
                raise NotImplementedError()
        return result_reward, result_action

    def ext_N_set(self, N_set):
        return (N_set[0]+2*int(N_set[0]*self.extends[1]/2), N_set[1]+2*int(N_set[1]*self.extends[0]/2))
    
def input_data(state):
    input_list = []
    for idx in range(800):
        i = idx//int(20)
        j = idx%int(20)
        phi_action = (i/40)%1
        theta_action = (j/20)%1
        actions = np.array([phi_action, theta_action, 0.1, 0.1])
        input = torch.tensor(np.concatenate((state, actions))).float().to(device)
        input_list.append(torch.unsqueeze(input, 0))
    total_input = torch.concat(input_list, dim=0)
    return total_input

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FOLDER_PATH = "C:/Users/dlgkr/Downloads/train0319_1/"
hidden_dim = 4096

model_paths = os.listdir(FOLDER_PATH)
models = []
for model_path in model_paths:
    model = QValueNet_CNN_B1(input_dim=1010, hidden_dim=hidden_dim, activation=nn.ELU, dropout=0.15).to(device)
    checkpoint = torch.load(FOLDER_PATH+model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("MODEL LOADED : %s"%(model_path))
    models.append(model)

In [ ]:
TESTSET_PATH = r"C:\Users\dlgkr\OneDrive\Desktop\code\astronomy\asteroid_AI\data\pole_axis_RL_data_batches\unrolled\RL_domain1\data_pole_axis_RL_preset_batch_4.npy"

map_modifier = RewardMapModifier(extends=(0, 0), blur_coef=(3, 2)) #if you use CNN, do not use extend method

testset = np.load(TESTSET_PATH)
testset_len = int(testset[0, 0])
test_img_num = 500 #10
test_img_idx_choice = np.random.choice(np.arange(0, (testset_len-1)//800), size=test_img_num, replace=False)
dataset_img_idx = np.full(testset_len, False)
for i in test_img_idx_choice:
    dataset_img_idx[i*800+1:(i+1)*800+1] = True

print("test_img_idx (in RL_preset_batch_2) :", test_img_idx_choice)
print("--------------------------------")
print("")

test_img_data = testset[dataset_img_idx, :].copy()
del testset
gc.collect()

test_img_list = []
for i in range(test_img_num):
    test_img_list.append(test_img_data[i*800:(i+1)*800, -2].reshape((40, 20)).T)

for i in tqdm(range(len(test_img_list))):
    test_img_list[i], _ = map_modifier.operation(np.expand_dims(test_img_list[i], axis=-1), None, order=['extend_vert', 'extend_hori', 'blur'])
    gc.collect()

In [ ]:
class CustomLoss4(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        input_reshaped = input.reshape(-1, 20, 40)
        target_reshaped = target.reshape(-1, 20, 40)

        torch_MSE = nn.MSELoss()
        input_prop = input_reshaped
        target_prop = target_reshaped

        loss = torch_MSE(input_prop, target_prop)

        return loss
    
loss_fn = CustomLoss4()

In [ ]:
losses = np.zeros((len(models), test_img_num))
for i, model in enumerate(models):
    for j in tqdm(range(test_img_num)):
        state = test_img_data[j*800, :1006]
        
        model.eval()
        with torch.no_grad():
            input = input_data(state)
            rewards = model(input)
            pred = rewards.cpu().reshape(40, 20).T
        losses[i, j] = loss_fn(pred, torch.tensor(test_img_list[j][:, :, 0])).item()

### Fundamental Drawings

In [ ]:
end_epoch = 40

plt.plot(train_loss[1:end_epoch+1], label='train_loss')
plt.plot(test_loss[1:end_epoch+1], label='test_loss')
plt.legend()
ylim = plt.ylim()
plt.show()

for i in range(indv_losses.shape[0]):
    plt.plot(indv_losses[i, 1:end_epoch+1], alpha=0.1)
plt.ylim((-0.05, ylim[1]))
plt.show()

### Advanced Drawings

#### Test Loss (avg, med w/ std) (A)

In [ ]:
median = np.median(indv_losses, axis=0)
average = np.average(indv_losses, axis=0)
sigma = np.std(indv_losses, axis=0)

end_epoch = 50
sigma_ratio = 0.5
plt.plot(median[1:end_epoch+1], label='test_median')
plt.fill_between(np.arange(1, end_epoch+1),
                 median[1:end_epoch+1]-sigma_ratio*sigma[1:end_epoch+1],
                 median[1:end_epoch+1]+sigma_ratio*sigma[1:end_epoch+1], alpha=0.2)
plt.plot(average[1:end_epoch+1], label='test_average')
plt.plot(train_loss[1:end_epoch+1], linestyle='--', label='train_loss')
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curve for Testset")
plt.show()

#### Hist. for Loss Dist. (A, B)

In [ ]:
epoch_list = [3, 30, 40, 70] #A

epoch_list = [1, 3, 4, 5] #B
indv_losses = np.swapaxes(losses, 0, 1)

median = np.median(indv_losses, axis=0)
average = np.average(indv_losses, axis=0)

# starts with epoch = 1
epoch_list = [x-1 for x in epoch_list]

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

def plot_hist(indv_losses, epoch_list, lim=(False, False), color=colors):
    xlim = None
    ylim = None
    for epoch, color in zip(epoch_list, colors):
        sns.histplot(indv_losses[:, epoch], bins=40, kde=True, alpha=0.20, label="%02d"%(epoch+1), linewidth=0.5, element='step')
        xlim = plt.xlim() if xlim is None else xlim
        ylim = plt.ylim() if ylim is None else ylim
        plt.plot([average[epoch]]*2, ylim, color=color, alpha=0.5, linestyle='dashed')
        plt.plot([median[epoch]]*2, ylim, color=color, alpha=0.6, linestyle='dotted')
    if lim[0]: plt.xlim((xlim[0], min(xlim[1], 4)))
    if lim[1]: plt.ylim((ylim[0], min(ylim[1], 100)))
    plt.xlabel("Loss")
    plt.ylabel("Frequency")
    plt.title("Hist. for Loss Distribution at Each Step")
    plt.legend()
    plt.show()

plot_hist(indv_losses, epoch_list, lim=(True, True), color=colors)
plot_hist(indv_losses, epoch_list, lim=(False, False), color=colors)

thr=0.8
print("#"*30)
print("Ratio of Data under Loss=%.2f"%(thr))
for i in range(indv_losses.shape[1]):
    num_under = np.sum(indv_losses[:, i] < thr)
    print("Epoch %2d | %.1f%% (%3d/%3d)"%(i+1, 100*num_under/indv_losses.shape[0], num_under, indv_losses.shape[0]))